# Практичне заняття №3: юніт-тести у Python

Виконуйте комірки зверху вниз. Notebook містить навчальні файли всередині й створює їх в окремому тимчасовому каталозі, тому клонувати репозиторій не потрібно.

**Контракт:** `pixel` — звичайне ціле число Python у `[0, 255]`, `delta` — ціла зміна; результатом є сума з насиченням на обох межах.

Під час експериментів окремі тести навмисно падають: це очікуваний результат перевірки дефекту. На правильній початковій версії всі 8 готових випадків мають проходити.

Збережіть власну копію notebook, щоб редагувати тести та залишити відповіді.

In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import shutil
import sys
import tempfile

if importlib.util.find_spec("pytest") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest>=8,<10"])

LAB_DIR = Path(tempfile.mkdtemp(prefix="lecture_three_"))
print("Робочий каталог:", LAB_DIR)


In [ ]:
# Початкові навчальні файли. Повторний запуск цієї комірки відновлює приклади.
FILES = {'image_ops.py': '"""Навчальний приклад: зміна яскравості одного пікселя.\n\nКонтракт: pixel — звичайне ціле число Python від 0 до 255,\ndelta — ціла зміна; результат обмежений діапазоном [0, 255].\nПоведінка для аргументів поза контрактом цим прикладом не задається.\n"""\n\n\ndef brighten(pixel: int, delta: int) -> int:\n    """Повернути яскравість після зміни з насиченням на обох межах."""\n    return max(0, min(255, pixel + delta))\n', 'test_image_ops.py': '"""Перший приклад зі слайда: Arrange, Act, Assert."""\n\nfrom image_ops import brighten\n\n\ndef test_brighten_increases_brightness():\n    # Arrange: незалежно визначені початкові дані.\n    pixel = 100\n    delta = 20\n\n    # Act: один виклик перевірюваної дії.\n    result = brighten(pixel, delta)\n\n    # Assert: очікування походить із контракту.\n    assert result == 120\n', 'test_boundaries.py': '"""Готовий навчальний зразок параметризації та граничних випадків.\n\nДля власної роботи створіть test_student.py з додатковими сценаріями.\nОчікування потрібно визначати з контракту, а не викликом brighten.\n"""\n\nimport pytest\n\nfrom image_ops import brighten\n\n\n@pytest.mark.parametrize(\n    "p,d,expected",\n    [\n        pytest.param(100, 20, 120, id="ordinary"),\n        pytest.param(250, 20, 255, id="upper_saturation"),\n        pytest.param(5, -20, 0, id="lower_saturation"),\n        pytest.param(0, 0, 0, id="zero_unchanged"),\n        pytest.param(255, 0, 255, id="maximum_unchanged"),\n        pytest.param(0, 1, 1, id="step_from_zero"),\n        pytest.param(255, -1, 254, id="step_from_maximum"),\n    ],\n)\ndef test_brighten_cases(p, d, expected):\n    assert brighten(p, d) == expected\n', 'demo_defects.py': '"""Порівняти той самий набір тестів на правильній і дефектних версіях.\n\nКожен експеримент створює тимчасову копію файлів та окремий процес Python.\nОригінальні image_ops.py й тести залишаються незмінними.\nЦе навчальний експеримент з трьома конкретними дефектами, а не повний\nінструмент мутаційного тестування.\n"""\n\nimport argparse\nimport importlib.util\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\n\n\nROOT = Path(__file__).resolve().parent\nIMPLEMENTATIONS = {\n    "correct": "return max(0, min(255, pixel + delta))",\n    "no_lower_bound": "return min(255, pixel + delta)",\n    "no_upper_bound": "return max(0, pixel + delta)",\n    "wrong_sign": "return max(0, min(255, pixel - delta))",\n}\n\n\ndef run_experiment(suite="all"):\n    if importlib.util.find_spec("pytest") is None:\n        raise SystemExit("Спочатку встановіть залежності: python -m pip install -r requirements.txt")\n    if suite == "first":\n        tests = [ROOT / "test_image_ops.py"]\n    elif suite == "student":\n        tests = [ROOT / "test_student.py"]\n        if not tests[0].is_file():\n            raise SystemExit("Спочатку створіть test_student.py зі своїми тестами.")\n    else:\n        tests = sorted(ROOT.glob("test_*.py"))\n\n    results = {}\n    for name, body in IMPLEMENTATIONS.items():\n        with tempfile.TemporaryDirectory(prefix="unit_tests_") as directory:\n            target = Path(directory)\n            (target / "image_ops.py").write_text(\n                "def brighten(pixel, delta):\\n    " + body + "\\n", encoding="utf-8"\n            )\n            for test in tests:\n                shutil.copyfile(test, target / test.name)\n            environment = os.environ.copy()\n            # Зовнішні pytest-плагіни не потрібні цьому навчальному прикладу.\n            environment["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"\n            environment.pop("PYTEST_ADDOPTS", None)\n            completed = subprocess.run(\n                [sys.executable, "-m", "pytest", "-q", "--tb=short", "--color=no",\n                 *[test.name for test in tests]],\n                cwd=target, text=True, capture_output=True, env=environment,\n            )\n            print(f"\\n=== {name} ===", flush=True)\n            print(completed.stdout, end="", flush=True)\n            if completed.stderr:\n                print(completed.stderr, end="", flush=True)\n            if completed.returncode not in (0, 1):\n                raise SystemExit(\n                    f"Код завершення {completed.returncode}: перевірте запуск "\n                    "або збирання тестів; це не результат виявлення дефекту."\n                )\n            results[name] = completed.returncode\n            if name == "correct" and completed.returncode != 0:\n                raise SystemExit(\n                    "Правильна версія не проходить набір: перевірте запуск і власні "\n                    "очікування перед експериментами з дефектами."\n                )\n\n    print("\\nПідсумок експерименту:")\n    for name, status in results.items():\n        if name == "correct":\n            label = "правильна версія проходить" if status == 0 else "перевірте власні очікування"\n        else:\n            label = "дефект виявлено" if status == 1 else "дефект НЕ виявлено"\n        print(f"{name}: {label}")\n\n    if results["correct"] != 0:\n        raise SystemExit("Правильна версія має проходити тести перед оцінкою дефектів.")\n    return results\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--suite", choices=["all", "first", "student"], default="all")\n    run_experiment(parser.parse_args().suite)\n'}
for name, source in FILES.items():
    (LAB_DIR / name).write_text(source, encoding="utf-8")
print("Створено:", ", ".join(FILES))

def run_python(*arguments, expected_exit=0):
    # Зміни 120 -> 121 можуть мати той самий розмір і timestamp у межах секунди.
    # Прибираємо локальний кеш, щоб перевіряти саме щойно відредагований файл.
    shutil.rmtree(LAB_DIR / "__pycache__", ignore_errors=True)
    environment = os.environ.copy()
    environment["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
    environment.pop("PYTEST_ADDOPTS", None)
    completed = subprocess.run(
        [sys.executable, "-B", *arguments], cwd=LAB_DIR,
        text=True, capture_output=True, env=environment,
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    if completed.returncode != expected_exit:
        raise RuntimeError(
            f"Очікувався код завершення {expected_exit}, отримано {completed.returncode}. "
            "Прочитайте звіт вище."
        )
    return completed.returncode


## 1. Очікування до запуску

Заповніть правий стовпець до виконання наступних комірок та поясніть, яке правило контракту застосовується.

| pixel | delta | Очікувана відповідь та пояснення |
|---:|---:|---|
| 100 | 20 | |
| 250 | 20 | |
| 5 | -20 | |

Додайте ще дві власні пари аргументів.

## 2. Один тест: Arrange, Act, Assert

Прочитайте приклад, знайдіть підготовку даних, виклик і перевірку, а потім запустіть тест.

In [ ]:
print((LAB_DIR / "test_image_ops.py").read_text(encoding="utf-8"))


In [ ]:
run_python("-m", "pytest", "-q", "test_image_ops.py")


### Навмисно неправильне очікування

Наступна комірка тимчасово змінює очікування на `121`, очікує невдачу тесту та відновлює початковий файл.

Після запуску поясніть: яке число повернула функція, яке число вимагав тест і чому очікування суперечить контракту.

In [ ]:
test_path = LAB_DIR / "test_image_ops.py"
original = test_path.read_text(encoding="utf-8")
try:
    test_path.write_text(original.replace("assert result == 120", "assert result == 121"), encoding="utf-8")
    run_python("-m", "pytest", "-q", "--tb=short", "test_image_ops.py", expected_exit=1)
finally:
    test_path.write_text(original, encoding="utf-8")
print("Початкове очікування відновлено.")


## 3. Граничні випадки та параметризація

Кожен рядок у `parametrize` створює окреме виконання. Порівняйте сценарії, а потім запустіть усі готові тести.

In [ ]:
print((LAB_DIR / "test_boundaries.py").read_text(encoding="utf-8"))


In [ ]:
run_python("-m", "pytest", "-v", "test_image_ops.py", "test_boundaries.py")


### Власні перевірки

У наступній комірці впишіть код своїх тестів усередині `STUDENT_SOURCE`.
Використайте `from image_ops import brighten`, назви функцій починайте з `test_` та задайте обґрунтовані очікування.
Перевірте межі, нульову зміну та щонайменше один новий випадок із поясненням.

Порожній шаблон не запускає власних тестів і не вважається виконаним завданням.

In [ ]:
STUDENT_SOURCE = """
# Напишіть тут власні тести за контрактом.
"""

if not any(line.strip() and not line.lstrip().startswith("#") for line in STUDENT_SOURCE.splitlines()):
    print("Власних тестів ще немає: заповніть STUDENT_SOURCE.")
else:
    (LAB_DIR / "test_student.py").write_text(STUDENT_SOURCE, encoding="utf-8")
    run_python("-m", "pytest", "-v", "test_student.py")


## 4. Чи помічають тести дефекти

Спочатку перевіримо **лише перший звичайний тест** на правильній версії та трьох навмисних дефектах.
Перед запуском передбачте, які зміни він пропустить. Експеримент використовує тимчасові копії й не змінює початкові файли.

In [ ]:
run_python("demo_defects.py", "--suite", "first")


### Повний набір

Порівняйте результати з першим тестом. Невдачі на дефектних реалізаціях очікувані та означають, що дефект виявлено.

In [ ]:
run_python("demo_defects.py")


### Перевірка власного набору

Після заповнення власних тестів запустіть наступну комірку й заповніть таблицю.

| Дефект | Прогноз | Тест, який упав | Порушене правило |
|---|---|---|---|
| Відсутня нижня межа | | | |
| Відсутня верхня межа | | | |
| Неправильний знак зміни | | | |

Якщо дефект не виявлено, доповніть свій набір і повторіть експеримент.

In [ ]:
if (LAB_DIR / "test_student.py").exists():
    run_python("demo_defects.py", "--suite", "student")
else:
    print("Спочатку створіть свої тести в комірці STUDENT_SOURCE.")


## Фінальний запуск

Правильна функція має проходити всі готові та власні тести. Для здачі збережіть власну копію notebook з відповідями, тестами та заповненими таблицями.

In [ ]:
run_python("-m", "pytest", "-q")


## Джерела

- [Перший тест](https://docs.pytest.org/en/stable/getting-started.html).
- [Параметризація](https://docs.pytest.org/en/stable/how-to/parametrize.html).
- [Репозиторій заняття](https://github.com/mitikov/applied-math-compute/tree/main/Lecture_three).
